# XGBoost: Predicting Hard-Drive Failure
**IST 707 Final Project**

This notebook trains an XGBoost model to predict whether a hard drive will fail in the next 30 days, using its SMART health readings from the last 30 days. It:

1. loads the training and calibration splits (the 2025 test split stays closed until final evaluation)
2. builds the model inputs, using the same feature rules as the random forest
3. trains a first XGBoost model with default settings and checks it on the calibration split
4. tunes XGBoost's settings with a randomized search over the **same** time-based folds as the random forest
5. saves the best model

**What XGBoost is:** it builds decision trees one after another, and each new tree focuses on the mistakes of the trees before it. It's usually one of the strongest methods on table-shaped data like ours, and it handles missing values by itself.

## 1. Libraries
Imports everything the notebook uses:
- **pandas / numpy:** loading and handling the data
- **scikit-learn:** the pipeline pieces (`Pipeline`, `ColumnTransformer`, `FunctionTransformer`, `OneHotEncoder`), the tuning tool (`RandomizedSearchCV`) and the scoring metrics
- **xgboost:** the `XGBClassifier` model. Install it once with `pip install xgboost`.
- **time / joblib:** timing the search and saving the final model to a file

`SimpleImputer` and `RandomForestClassifier` are also imported but not used in this notebook, because XGBoost doesn't need blanks filled.

In [2]:
import numpy as np
import pandas as pd
import time
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)
from xgboost import XGBClassifier

## 2. Loading the Data
Loads the two splits used for training and checking. **The test split is not loaded.**

- **`modeling_train.parquet`:** Jan 2023 – Oct 2024. 169,827 drive-windows, 8,087 failures. Healthy windows were down-sampled to 20 per failure, so failures are 1 in 21.
- **`modeling_calib.parquet`:** Nov – Dec 2024. 297,009 drive-windows, 328 failures, at the real failure rate of about 1 in 905.

For each split the cell creates:
- **`X_..._raw`:** every column except the answer
- **`y_...`:** the answer, `label` (1 = the drive fails in the next 30 days)
- **`meta_...`:** serial number, window number and model. These are **not** model inputs; they're used to build the time-based folds.

In [3]:
train = pd.read_parquet("../../modeling/modeling_train.parquet")
calib = pd.read_parquet("../../modeling/modeling_calib.parquet")

X_train_raw, y_train = train.drop(columns="label"), train["label"].astype(int)
X_calib_raw, y_calib = calib.drop(columns="label"), calib["label"].astype(int)

meta_train = train[["serial_number", "window", "model"]].reset_index(drop=True)
meta_calib = calib[["serial_number", "window", "model"]].reset_index(drop=True)

print("Training shape:", X_train_raw.shape)
print("Calibration shape:", X_calib_raw.shape)

display(X_train_raw.head())
display(X_calib_raw.head())

display(y_train.head())
display(y_calib.head())

Training shape: (170205, 93)
Calibration shape: (297879, 93)


,serial_number,window,smart_10_normalized_first,smart_10_normalized_last,smart_10_raw_first,smart_10_raw_last,smart_12_normalized_first,smart_12_normalized_last,smart_12_raw_first,smart_12_raw_last,...,smart_3_raw_delta,smart_4_normalized_delta,smart_4_raw_delta,smart_5_normalized_delta,smart_5_raw_delta,smart_7_normalized_delta,smart_7_raw_delta,smart_9_normalized_delta,smart_9_raw_delta,drive_days_to_date
0,043d3be09f4a0010,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30
1,0564f6f3fab90010,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30
2,1030A00KF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,5.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,696.0,30
3,1030A014F97G,0,100.0,100.0,0.0,0.0,100.0,100.0,4.0,5.0,...,-37.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.0,696.0,30
4,1040A00BF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,698.0,30


,serial_number,window,smart_10_normalized_first,smart_10_normalized_last,smart_10_raw_first,smart_10_raw_last,smart_12_normalized_first,smart_12_normalized_last,smart_12_raw_first,smart_12_raw_last,...,smart_3_raw_delta,smart_4_normalized_delta,smart_4_raw_delta,smart_5_normalized_delta,smart_5_raw_delta,smart_7_normalized_delta,smart_7_raw_delta,smart_9_normalized_delta,smart_9_raw_delta,drive_days_to_date
0,14A0A001F9RG,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,119.0,6
1,14B0A005F9RG,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,139.0,7
2,1PG6SGEV,22,100.0,100.0,0.0,0.0,100.0,100.0,8.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,78.0,5
3,1PG700ZV,22,100.0,100.0,0.0,0.0,100.0,100.0,10.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,94.0,5
4,1PG73SEV,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,171.0,8


0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64

0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64

## 3. Building the Model Inputs
Defines how the raw table is turned into model inputs. These are the **same rules as the random forest**, so the two models see the same information. This only sets rules; nothing is calculated from the data here.

- **`KEEP_RAW`:** the SMART attributes kept, as raw values only. The EDA showed the vendor-normalized values lose detail and partly encode the brand. S10 is left out because it is always 0.
- **`vendor()`:** reads the manufacturer (Seagate, Toshiba, WDC, HGST) from the model name.
- **`build_features()`:**
  - keeps the first, last and change values for each kept attribute
  - keeps S9 (power-on hours) as drive age
  - turns the unknown capacity of -1 into a blank (23 training rows)
  - keeps days observed and the vendor
- **`NUMERIC` / `CATEGORICAL`:** the lists of number columns and the one text column (vendor).
- **`features` / `vendor_onehot`:** wrap these rules so they can sit inside the pipeline. The one-hot encoder turns vendor into yes/no columns, using a fixed vendor list so train and calibration always get the same columns.

In [4]:
KEEP_RAW = [1, 3, 4, 5, 7, 12, 192, 193, 194, 197, 198, 199]   # raw only; S9 -> age, S10 always 0
VENDORS = ["Seagate", "Toshiba", "WDC", "HGST", "Other"]


def vendor(model):
    m = str(model)
    if m.startswith("ST"): return "Seagate"
    if m.startswith("TOSHIBA"): return "Toshiba"
    if m.startswith(("WDC", "WUH")): return "WDC"
    if m.startswith(("HGST", "Hitachi")): return "HGST"
    return "Other"

## This Function Keeps raw SMART values only. Drops useless columns. Calculates drive age through power-on hours and cleans the storage column.
def build_features(df):
    X = pd.DataFrame(index=df.index)
    for i in KEEP_RAW:
        for s in ("first", "last", "delta"):
            X[f"s{i}_{s}"] = df[f"smart_{i}_raw_{s}"].astype("float64")
    X["s9_last"] = df["smart_9_raw_last"].astype("float64")          
    X["capacity_tb"] = (df["capacity_bytes"] / 1e12).round(0).where(df["capacity_bytes"] > 0)   # -1 = unknown -> NaN
    X["days_observed"] = df["days_observed"].astype("float64")
    X["vendor"] = df["model"].map(vendor)
    return X

NUMERIC = [f"s{i}_{s}" for i in KEEP_RAW for s in ("first", "last", "delta")] + \
          ["s9_last", "capacity_tb", "days_observed"]

CATEGORICAL = ["vendor"]

features = FunctionTransformer(build_features)
vendor_onehot = OneHotEncoder(categories=[VENDORS], handle_unknown="ignore", sparse_output=False)

## 4. Preprocessing Pipeline
Chains the preprocessing steps into one object, `xgb_preprocess`:

1. **features:** applies `build_features` to the raw table
2. **columns:**
   - **numbers:** passed through **unchanged, blanks included**. XGBoost handles missing values itself: at every split it learns whether drives with a missing value should go left or right. The blanks come from specific drive models (S197 on two Toshiba models, S193 on one laptop model) and from unknown capacity.
   - **vendor:** one-hot encoded

**The difference from the random forest:** the random forest pipeline fills blanks with the training median and adds "was missing" columns, because scikit-learn's forest needs complete data. XGBoost doesn't, so filling them would only throw information away.

In [5]:
xgb_preprocess = Pipeline([
    ("features", features),
    ("columns", ColumnTransformer([
        ("numeric", "passthrough", NUMERIC),
        ("vendor", vendor_onehot, CATEGORICAL),
    ], verbose_feature_names_out=False)),
]).set_output(transform="pandas")

## 5. First XGBoost Model (Before Tuning)
Builds the full model, preprocessing followed by XGBoost. Only a few settings are fixed here; everything else uses XGBoost's defaults (100 trees, depth 6, learning rate 0.3):

- **`objective="binary:logistic"`:** a yes/no prediction that outputs a failure probability
- **`tree_method="hist"`:** the fast CPU method, which groups values into bins before searching for splits
- **`eval_metric="aucpr"`:** XGBoost's internal score is PR-AUC, our main metric
- **`n_jobs=-1`:** uses all CPU threads
- **`random_state=42`:** makes results repeatable

No extra weight is given to failures in this first model (`scale_pos_weight` keeps its default of 1). The tuning step tries 1, 5 and 20.

In [6]:
xgb_model = Pipeline([
    ("preprocess", xgb_preprocess),
    ("classifier", XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",        # fast CPU method
        eval_metric="aucpr",       # PR-AUC, matches our main metric
        n_jobs=-1,                 # all 8 threads per model
        random_state=42,
    )),
])

### 5.1 Training the First Model
Fits the whole pipeline on the training split: the preprocessing builds the features, then XGBoost builds its trees one after another.

In [7]:
xgb_model.fit(X_train_raw, y_train)
joblib.dump(xgb_model, "../artifacts/xgb_base.joblib")

['../artifacts/xgb_base.joblib']

### 5.2 Checking the First Model on the Calibration Split
Predicts a failure probability for every calibration window, then turns those probabilities into yes/no predictions at thresholds from 0.0 to 1.0. At each threshold it records:

- **PR-AUC (AP):** how well the model *ranks* failing drives above healthy ones, over all thresholds. This is our main metric, and random guessing scores 0.0011.
- **ROC-AUC:** another ranking score (0.5 = coin flip, 1.0 = perfect). It looks good easily when failures are rare, so it's secondary.
- **Precision:** of the drives flagged, the share that actually failed
- **Recall:** of the drives that failed, the share that were flagged
- **F1:** one number balancing precision and recall
- **Brier score:** how close the probabilities are to what actually happened (lower is better)

PR-AUC, ROC-AUC and the Brier score don't depend on the threshold, which is why they're the same in every row. Only precision, recall and F1 change.

**Result (untuned XGBoost):**
- **Ranking:** PR-AUC is **0.0927**. That's slightly below the untuned random forest (0.0995), well above logistic regression (0.082) and far above the SMART 5 rule (0.0073) and random (0.0011). ROC-AUC is 0.883.
- **Thresholds:** at 0.9, precision is 14.3% and recall 24.7%, the best F1 (0.181). At the default 0.5, precision is only 6.3%.
- **Calibration:** the Brier score (0.0076) is better than the random forest's (0.0119), because without extra failure weighting the probabilities are less inflated. They're still too high for the real failure rate, so a threshold has to be chosen deliberately and the probabilities recalibrated later.

In [8]:
results=[]
y_prob = xgb_model.predict_proba(X_calib_raw)[:, 1]

# Default decision threshold
for x in [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]:
    y_pred = (y_prob >= x).astype(int)

    results.append({
        "Threshold": x,
        "Model": "XGBoost",
        "PR-AUC (AP)": average_precision_score(y_calib, y_prob),
        "ROC-AUC": roc_auc_score(y_calib, y_prob),
        "Precision": precision_score(
            y_calib, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_calib, y_pred, zero_division=0
        ),
        "F1": f1_score(y_calib, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_calib, y_prob)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

,Threshold,Model,PR-AUC (AP),ROC-AUC,Precision,Recall,F1,Brier Score
0,0.0,XGBoost,0.0898,0.8879,0.0011,1.0000,0.0022,0.0076
1,0.1,XGBoost,0.0898,0.8879,0.0232,0.7043,0.0448,0.0076
2,0.2,XGBoost,0.0898,0.8879,0.0350,0.6646,0.0665,0.0076
3,0.3,XGBoost,0.0898,0.8879,0.0459,0.6372,0.0857,0.0076
4,0.4,XGBoost,0.0898,0.8879,0.0541,0.5945,0.0991,0.0076
5,0.5,XGBoost,0.0898,0.8879,0.0618,0.5549,0.1112,0.0076
6,0.6,XGBoost,0.0898,0.8879,0.0733,0.5366,0.1290,0.0076
7,0.7,XGBoost,0.0898,0.8879,0.0855,0.4787,0.1451,0.0076
8,0.8,XGBoost,0.0898,0.8879,0.1050,0.3811,0.1646,0.0076
9,0.9,XGBoost,0.0898,0.8879,0.1329,0.2439,0.1720,0.0076


## 6. Tuning XGBoost
### 6.1 Search Setup and Time-Based Folds
Sets up three things for tuning:

- **`xgb_pipeline`:** the same preprocessing plus an XGBoost model whose settings will be searched
- **`param_distributions`:** the settings to try:
  - **`n_estimators`:** how many trees are built in sequence
  - **`learning_rate`:** how much each new tree corrects the previous ones. Smaller is more careful but needs more trees.
  - **`max_depth`:** how deep each tree can grow. Boosted trees are usually shallow.
  - **`min_child_weight`:** the minimum amount of data in a leaf, similar to the random forest's `min_samples_leaf`
  - **`subsample` / `colsample_bytree`:** the share of rows and of columns each tree sees, which adds variety and limits memorizing
  - **`gamma`:** the minimum improvement needed before a split is made
  - **`reg_lambda`:** a penalty that keeps leaf values small, to reduce overfitting
  - **`scale_pos_weight`:** extra weight on failures: 1 = none, 20 = matches train's 20:1 ratio

  Together that's 26,244 combinations.
- **`time_folds()`:** builds the **same 4 time-based folds as the random forest**, so both models are compared on the same months. Each fold trains on earlier months and validates on later months, with a one-window gap in between, because each label looks one window ahead.

The printout shows the 4 folds growing: training uses more months each time, and each validation block covers the next few months.

In [13]:
xgb_pipeline = Pipeline([
    ("preprocess", xgb_preprocess),
    ("classifier", XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",        # fast CPU method
        eval_metric="aucpr",       # PR-AUC, matches our main metric
        n_jobs=-1,                 # all 8 threads per model
        random_state=42,
    )),
])

param_distributions = {
    "classifier__n_estimators": [50,75,100, 150,],
    "classifier__learning_rate": [0.04, 0.05, 0.06],     # how big each tree's correction is
    "classifier__max_depth": [4, 5, 6, 7],               # boosted trees are shallow
    "classifier__min_child_weight": [0.1,0.5,1],          # minimum weight in a leaf (like min_samples_leaf)
    "classifier__subsample": [0.6, 0.7, 0.8],            # share of rows each tree sees
    "classifier__colsample_bytree": [0.7, 0.8, 0.9],     # share of columns each tree sees
    "classifier__gamma": [2,3,4,5],                      # minimum gain needed to make a split
    "classifier__reg_lambda": [0.1,0.5,1],                # penalty that keeps leaf values small
    "classifier__scale_pos_weight": [0.1, 0.5, 1],          # extra weight on failures (train is 20:1)
}


## Creating a cross validation set.
def time_folds(meta, n_splits=4, gap=1):
    """Expanding-window folds by calendar window. Skips `gap` windows between train and validation,
    because a window's label looks one window ahead."""
    w = meta["window"].to_numpy()
    windows = np.sort(np.unique(w))
    blocks = np.array_split(windows, n_splits + 1)
    folds = []
    for k in range(1, n_splits + 1):
        valid_w = blocks[k]
        train_w = windows[windows < valid_w.min() - gap]
        folds.append((np.flatnonzero(np.isin(w, train_w)), np.flatnonzero(np.isin(w, valid_w))))
    return folds

cv_folds = time_folds(meta_train)
for k, (tr_idx, va_idx) in enumerate(cv_folds):
    print(f"fold {k}: train {len(tr_idx):,} rows -> validate {len(va_idx):,} rows")


fold 0: train 28,001 rows -> validate 37,252 rows
fold 1: train 64,564 rows -> validate 31,291 rows
fold 2: train 95,661 rows -> validate 32,971 rows
fold 3: train 128,294 rows -> validate 33,683 rows


### 6.2 Untuned Model on the Time Folds
Scores the untuned XGBoost model from section 5 (`xgb_model`) on the same 4 time-based folds the search uses, with the same four metrics: precision, recall, F1 and PR-AUC. This is the "before" number for tuning.

**Why this is needed.** Section 5.2 scored the untuned XGBoost model on the calibration split, where failures are about 1 in 905. The search scores on training folds, where they are 1 in 21. PR-AUC depends heavily on the failure rate, so the 5.2 score (0.0927) can't be compared with the search scores in 6.3. Scoring the untuned XGBoost model on the same folds the search uses leaves tuning as the only difference between them.

- **`cross_validate`:** trains the untuned model once per fold, scores it on that fold's validation months, and reports the mean and std of each metric over the 4 folds.
- **`scoring`:** the same four metrics the search uses, so the before and after numbers are measured the same way.
- **`n_jobs=1`:** runs one fold at a time, as the search does, because each model already uses every CPU thread.

**Result:** *to be filled in once the cell has run.*
- **Ranking:** mean PR-AUC over the 4 folds (std), and why it is far above the 0.0927 in section 5.2.
- **At the default 0.5 threshold:** precision, recall and F1, each with its std.
- **Time:** mean fit time per fold.

In [10]:
# cross_validate isn't in the section 1 imports, so it is imported here
from sklearn.model_selection import cross_validate

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "pr_auc": "average_precision",
}

# n_jobs=1: one fold at a time, as the search does; each model already uses every CPU thread
start = time.time()
cv_untuned_results = cross_validate(xgb_model, X_train_raw, y_train, cv=cv_folds, scoring=scoring, n_jobs=1)
print(f"Cross-validation took {(time.time() - start) / 60:.1f} minutes")
print(" ")

for i in cv_untuned_results:
    print(f'{i} mean: {np.mean(cv_untuned_results[i])}')
    print(f'{i} standard deviation: {np.std(cv_untuned_results[i])}')
    print(' ')


Cross-validation took 0.1 minutes
 
fit_time mean: 1.0658104419708252
fit_time standard deviation: 0.35582272663857356
 
score_time mean: 0.17771369218826294
score_time standard deviation: 0.00978927743549957
 
test_precision mean: 0.7210643727707016
test_precision standard deviation: 0.02867036681671596
 
test_recall mean: 0.48109270454366393
test_recall standard deviation: 0.02929506192368786
 
test_f1 mean: 0.5769681564298296
test_f1 standard deviation: 0.0291745915456553
 
test_pr_auc mean: 0.5685450828866911
test_pr_auc standard deviation: 0.04041857122083054
 


### 6.3 Randomized Search, Results and Saving the Best Model
- **Randomized search:** tries **20 random combinations**. Each is trained once per fold, giving **80 fits**. `n_jobs=1` trains one model at a time, because each model already uses every CPU thread.
- **Scoring:** every fit is scored on four metrics:
  - **PR-AUC** (`average_precision`), which picks the winner (`refit="pr_auc"`)
  - **precision, recall and F1**, at XGBoost's default cut-off of 0.5
- **Refit:** the best combination by PR-AUC is retrained on all of the training data.
- **Results table:** each metric's average over the 4 folds (`PR_AUC`, `Precision`, `Recall`, `F1`), how much it varies across folds (`..._std`), the training time, the settings tried, and a rank (1 = best PR-AUC). The table is saved to `xgb_search_results_ranked.csv`.
- **Save:** the best model is saved to `xgb_best.joblib`, so it can be reloaded without retraining.

These scores come from training folds, where failures are 1 in 21, so they're much higher than the same metrics on the calibration split. Use them only to compare settings with each other. Precision, recall and F1 also depend on the 0.5 cut-off, so PR-AUC is the fairer comparison.

**Result:**
- **Time:** the search took **5.6 minutes**.
- **Best settings:** **300 trees, `learning_rate=0.05`, `max_depth=4`, `min_child_weight=1`, `subsample=0.7`, `colsample_bytree=0.7`, `gamma=5`, `reg_lambda=1`, `scale_pos_weight=1`**, with a mean time-CV PR-AUC of **0.595** (std 0.030).
- **Close calls:** the top 10 settings are within 0.01 of each other, less than one std apart, so they perform about the same.

In [14]:
search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring={
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "pr_auc": "average_precision",
    },
    cv=cv_folds,                   # same time-based folds as the random forest
    n_jobs=1,                      # one model at a time; each uses all threads
    refit="pr_auc",
    random_state=42,
    verbose=2,
    return_train_score=False,
)


start = time.time()
search.fit(X_train_raw, y_train)
print(f"\nSearch took {(time.time() - start) / 60:.1f} minutes")

cv_results = pd.DataFrame(search.cv_results_)

# All four metrics, averaged across the CV folds (column names follow the scoring keys)
results = pd.DataFrame({
    "PR_AUC":         cv_results["mean_test_pr_auc"],
    "PR_AUC_std":     cv_results["std_test_pr_auc"],
    "Precision":      cv_results["mean_test_precision"],
    "Precision_std":  cv_results["std_test_precision"],
    "Recall":         cv_results["mean_test_recall"],
    "Recall_std":     cv_results["std_test_recall"],
    "F1":             cv_results["mean_test_f1"],
    "F1_std":         cv_results["std_test_f1"],
    "mean_fit_time":  cv_results["mean_fit_time"],
})

for col in cv_results.columns:
    if col.startswith("param_"):
        results[col] = cv_results[col]

# Rank by PR-AUC (the same ranking the search used to pick the best model)
results["PR_AUC_rank"] = cv_results["rank_test_pr_auc"]
results = results.sort_values("PR_AUC_rank").reset_index(drop=True)

print("Best PR-AUC (time-CV on train):", round(search.best_score_, 4))
print("Best parameters:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")


print("\nTop 10 models ranked by PR-AUC:")
display(results.head(10))
results.to_csv("xgb_search_results_ranked.csv", index=False)

# ---------------------------------------------------------------- 6. Save the best model
best_xgb = search.best_estimator_
joblib.dump(best_xgb, "../artifacts/xgb_best.joblib")


Fitting 4 folds for each of 20 candidates, totalling 80 fits
[CV] END classifier__colsample_bytree=0.8, classifier__gamma=2, classifier__learning_rate=0.04, classifier__max_depth=4, classifier__min_child_weight=1, classifier__n_estimators=75, classifier__reg_lambda=0.1, classifier__scale_pos_weight=0.1, classifier__subsample=0.6; total time=   0.5s
[CV] END classifier__colsample_bytree=0.8, classifier__gamma=2, classifier__learning_rate=0.04, classifier__max_depth=4, classifier__min_child_weight=1, classifier__n_estimators=75, classifier__reg_lambda=0.1, classifier__scale_pos_weight=0.1, classifier__subsample=0.6; total time=   0.7s
[CV] END classifier__colsample_bytree=0.8, classifier__gamma=2, classifier__learning_rate=0.04, classifier__max_depth=4, classifier__min_child_weight=1, classifier__n_estimators=75, classifier__reg_lambda=0.1, classifier__scale_pos_weight=0.1, classifier__subsample=0.6; total time=   0.9s
[CV] END classifier__colsample_bytree=0.8, classifier__gamma=2, class

,PR_AUC,PR_AUC_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,mean_fit_time,param_classifier__subsample,param_classifier__scale_pos_weight,param_classifier__reg_lambda,param_classifier__n_estimators,param_classifier__min_child_weight,param_classifier__max_depth,param_classifier__learning_rate,param_classifier__gamma,param_classifier__colsample_bytree,PR_AUC_rank
0,0.593287,0.034367,0.807272,0.021201,0.405787,0.017923,0.540013,0.020042,1.554516,0.8,0.5,1.0,150,0.5,6,0.04,2,0.7,1
1,0.591117,0.032665,0.743544,0.027356,0.486100,0.024044,0.587752,0.025000,0.957845,0.7,1.0,1.0,75,1.0,6,0.06,4,0.7,2
2,0.590338,0.034631,0.743375,0.028750,0.481512,0.023063,0.584360,0.024954,1.044543,0.6,1.0,0.5,75,1.0,6,0.05,3,0.9,3
3,0.590098,0.032429,0.799799,0.019783,0.418807,0.015633,0.549697,0.017660,1.452078,0.6,0.5,0.1,100,0.5,7,0.05,2,0.7,4
4,0.589012,0.030570,0.744675,0.027556,0.484026,0.024134,0.586510,0.024283,0.895086,0.7,1.0,0.1,75,1.0,5,0.06,3,0.9,5
5,0.588604,0.034218,0.807774,0.015285,0.394106,0.016695,0.529678,0.018189,1.023868,0.8,0.5,1.0,75,1.0,6,0.05,4,0.9,6
6,0.585195,0.033158,0.810903,0.021182,0.381646,0.014726,0.518987,0.017710,0.993358,0.7,0.5,0.5,100,0.1,4,0.05,5,0.9,7
7,0.584698,0.032849,0.812374,0.020605,0.383534,0.012922,0.521001,0.015225,0.999941,0.6,0.5,0.5,75,0.5,5,0.04,2,0.8,8
8,0.583876,0.032403,0.826633,0.018367,0.358767,0.009871,0.500354,0.012728,0.872976,0.7,0.5,0.5,50,0.5,6,0.04,2,0.7,9
9,0.581759,0.033156,0.911291,0.005697,0.159741,0.018086,0.271426,0.026296,1.042801,0.8,0.1,0.5,100,0.1,5,0.05,2,0.7,10


['../artifacts/xgb_best.joblib']

## 7. Final Scoring on the Calibration Split
Scores the tuned XGBoost model from 6.3 **once** on the calibration split and saves its predictions for the model comparison. Nothing is tuned here.

- **Why calib:** the search chose its settings on training folds, where failures are 1 in 21. The calibration split (November to December 2024) has failures at the real rate of about 1 in 905, comes after every training month, and was never used to fit or tune this model. It shows how the tuned model ranks drives under conditions like real use, and every model is scored on exactly the same rows, so the scores can be compared directly.
- **Why only once:** calib holds only 328 failures. Scoring it repeatedly and adjusting the model each time would tune the model to those 328 drives. The 2025 test split stays closed until the final model is chosen.
- **The model:** loaded from `../artifacts/xgb_best.joblib`, the same file the final test will load, so the search doesn't need to be re-run.
- **Saved file:** `xgb_calib_preds.csv` in this folder, one row per calibration window, with the columns `serial_number`, `window`, `label` and `prob`. The comparison notebook builds every model's PR-AUC and replacement-budget results from these files, and calibrates the winning model from them.

**Result:** *to be filled in once the cell has run.*
- **Ranking:** PR-AUC on calib, against 0.0927 for the untuned XGBoost model in section 5.2. Both are on the calibration split, so this is a direct before-and-after comparison.
- **Saved:** row count of `xgb_calib_preds.csv` (297,009 expected) and its number of failures (328 expected).

In [15]:
xgb_final = joblib.load("../artifacts/xgb_best.joblib")

calib_scores = xgb_final.predict_proba(X_calib_raw)[:, 1]

calib_preds = pd.DataFrame({
    'serial_number': meta_calib['serial_number'].copy(),
    'window': meta_calib['window'].copy(),
    'label': y_calib.to_numpy(),
    'prob': calib_scores
})

calib_preds.to_csv('xgb_calib_preds.csv', index = False)

print(f"Average precision score: {average_precision_score(calib_preds['label'], calib_preds['prob'])}")
print(f'Row count: {len(calib_preds)}')
print(f"Failure count: {calib_preds['label'].sum()}")


Average precision score: 0.12511416230978514
Row count: 297879
Failure count: 328
